# 🎯 Module 07: Vector Databases and Retrieval

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 2 — Retrieval and RAG Foundations  
> **Module:** 07 — Vector Databases and Retrieval  
> **Planned Date:** July 16, 2026  
> **Level:** Beginner → Intermediate → Advanced  
> **Prerequisites:** Python, Modules 01–06  
> **Previous:** 06 — Embeddings and Semantic Search  
> **Next:** 08 — RAG Fundamentals

---

We know how to embed text and compare vectors. Now we need to **store, index, filter, and retrieve them at scale**.

In this module we'll cover:

- What a **vector database** is
- **Vectors, records, collections, metadata**
- **Traditional DB vs vector DB**
- **Similarity metrics** & **ANN** indexes
- **Top-K retrieval** & **metadata filters**
- **ChromaDB** hands-on
- **Retrieval quality** — Precision@K, Recall@K, MRR, latency
- **ChromaDB vs Qdrant vs pgvector**
- A **practical lab**: knowledge base retrieval system
- Common mistakes & best practices

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain what a **vector database** is & why it's useful
2. Understand **vectors, records, collections, payloads, metadata**
3. Compare **regular vs vector** databases
4. Store & retrieve **text embeddings**
5. Understand **similarity metrics** and **ANN search**
6. Implement **top-k retrieval**
7. Apply **metadata filters**
8. Choose between **ChromaDB, Qdrant, PostgreSQL + pgvector**
9. **Evaluate retrieval quality** and **latency**
10. Build a **reusable retrieval layer** for RAG

---
## ⚙️ 0. Setup & Imports

This notebook is **safe to run without an API key**. ChromaDB runs locally, and we include a fallback embedding function so the notebook always works.

In [1]:
import os
import json
import hashlib
import numpy as np
from pathlib import Path

np.random.seed(42)
print('✅ Setup ready.')

# Check for chromadb
try:
    import chromadb
    print(f'ChromaDB version: {chromadb.__version__}')
    HAS_CHROMA = True
except ImportError:
    print('ChromaDB not installed. Run: pip install chromadb')
    HAS_CHROMA = False

HAS_KEY = bool(os.getenv('OPENAI_API_KEY'))
print(f'OPENAI_API_KEY present: {HAS_KEY}')

✅ Setup ready.
ChromaDB not installed. Run: pip install chromadb
OPENAI_API_KEY present: False


---
## 🧠 1. What Is a Vector Database?

A **vector database** stores and searches **numerical vectors**, often alongside the **text** and **metadata** associated with those vectors.

### Why?

For 100,000 documents, computing cosine similarity between a query and every document becomes **expensive**.

A vector DB provides **specialized storage + indexing + search** for vectors.

### Example retrieval record

```text
Record
├── ID: doc_001
├── Vector: [0.12, -0.43, 0.81, ...]
├── Text: "How to reset an account password"
└── Metadata
    ├── category: "support"
    └── source: "help-center"
```

In [2]:
# Visualize the structure of a retrieval record
record = {
    'id': 'doc_001',
    'vector': [0.12, -0.43, 0.81, 0.27],
    'text': 'How to reset an account password',
    'metadata': {
        'category': 'support',
        'source': 'help-center',
    },
}

print(json.dumps(record, indent=2))

{
  "id": "doc_001",
  "vector": [
    0.12,
    -0.43,
    0.81,
    0.27
  ],
  "text": "How to reset an account password",
  "metadata": {
    "category": "support",
    "source": "help-center"
  }
}


---
## 🆚 2. Traditional Database vs Vector Database

| Feature | Traditional DB | Vector DB |
| --- | --- | --- |
| Primary purpose | Store & query structured data | Store & search vectors |
| Common query | Find by ID | Find similar to a vector |
| Exact matching | ✅ Excellent | ⚠️ Not its main purpose |
| Semantic retrieval | Requires extra methods | ✅ Core capability |
| Filtering | SQL / DB-specific | Metadata filters |
| Examples | PostgreSQL, MySQL, SQL Server | Qdrant, ChromaDB, Milvus |
| Typical AI use | Users, orders, transactions | Embeddings & retrieval |

> ⚠️ **Categories overlap.** PostgreSQL + **pgvector** can do BOTH relational data AND vector search in one database.

> **A vector DB is not automatically the best solution.** Choose based on **data size, filtering, infrastructure, security, latency, operations**.

---
## 🧩 3. Core Concepts

### 3.1 Vector
A sequence of numbers representing an item.

```python
embedding = [0.12, -0.43, 0.81, 0.27]
```

### 3.2 Collection
A logical group of records.

```text
company_documents
├── HR policies
├── Technical documentation
├── Product documentation
└── Support articles
```

### 3.3 Metadata / Payload
Additional info about a document.

```python
metadata = {
    'category': 'hr',
    'department': 'human_resources',
    'source': 'employee_handbook.pdf',
    'page': 12,
}
```

### 3.4 Similarity Metric

| Metric | Interpretation |
| --- | --- |
| Cosine similarity | Compares **direction** |
| Dot product | Aligned values; magnitude matters |
| Euclidean distance | Straight-line distance |

### 3.5 Top-K Retrieval
Returns the best-ranked **k** candidates.

### 3.6 Vector Index / ANN
Speeds up nearest-neighbor search by trading some exactness for performance at scale.

---
## 🔄 4. How Vector Retrieval Works

### Document ingestion

```text
Documents → Split into Chunks → Generate Embeddings
                                      │
                                      ▼
                            Store Vectors + Metadata
                                      │
                                      ▼
                              Vector Database
```

### Query-time retrieval

```text
User Query → Generate Query Embedding
                         │
                         ▼
               Similarity Search + Filters
                         │
                         ▼
                  Ranked Results
                         │
                         ▼
             Application / Future RAG System
```

> ⚠️ Document & query embeddings must be **compatible**. Switching to an incompatible embedding model requires **re-embedding** the stored documents.

---
## 🛠️ 5. Hands-On with ChromaDB

We start with **ChromaDB** — simple, local, easy to learn.

### Install
```bash
pip install chromadb openai python-dotenv pytest
```

### Key operations

- `PersistentClient` — creates a local persistent client
- `get_or_create_collection` — opens or creates a collection
- `upsert` — inserts or updates records with the same ID
- `query` — searches with a text or vector query
- `where` — metadata filter
- `n_results` — top-k

In [3]:
# Create a persistent ChromaDB collection and store some documents
if HAS_CHROMA:
    import chromadb
    from pathlib import Path

    # Use a local folder inside this module
    CHROMA_PATH = SCRIPT_DIR / 'chroma_data'
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))

    # Fresh start for the demo
    try:
        client.delete_collection('knowledge_base')
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name='knowledge_base',
        metadata={'description': 'Learning vector retrieval'},
    )

    documents = [
        'Reset your password from account settings.',
        'Update payment information in your billing portal.',
        'Troubleshoot a server that is not responding.',
        'Recover access to an account when credentials are forgotten.',
    ]
    ids = ['doc_001', 'doc_002', 'doc_003', 'doc_004']
    metadatas = [
        {'category': 'support'},
        {'category': 'billing'},
        {'category': 'engineering'},
        {'category': 'support'},
    ]

    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
    )

    print('✅ Documents stored.')
    print('Collection count:', collection.count())
else:
    print('Skipped — install chromadb first (pip install chromadb).')

Skipped — install chromadb first (pip install chromadb).


In [4]:
# Retrieval function with metadata filtering
if HAS_CHROMA:
    def search_documents(query, top_k=3, category=None):
        if not query.strip():
            raise ValueError('Query cannot be empty.')
        if top_k < 1:
            raise ValueError('top_k must be at least 1.')

        where = {'category': category} if category else None

        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            where=where,
        )

        docs = results['documents'][0] or []
        ids_ = results['ids'][0] or []
        metas = results['metadatas'][0] or []
        dists = results['distances'][0] or []

        return [
            {'id': i, 'document': d, 'metadata': m, 'distance': dist}
            for i, d, m, dist in zip(ids_, docs, metas, dists)
        ]

    # Run a semantic query
    results = search_documents('I forgot my login password', top_k=3)
    print('Query: "I forgot my login password"\n')
    for r in results:
        print(f"  id={r['id']}  dist={r['distance']:.4f}  {r['document']}")
else:
    print('Skipped — ChromaDB not installed.')

Skipped — ChromaDB not installed.


In [5]:
# Metadata filtering demo — restrict to support category
if HAS_CHROMA:
    results = search_documents(
        'I cannot access my account',
        top_k=3,
        category='support',
    )
    print('Query: "I cannot access my account"  (category=support)\n')
    for r in results:
        print(f"  id={r['id']}  dist={r['distance']:.4f}  "
              f"cat={r['metadata']['category']}  {r['document']}")
else:
    print('Skipped — ChromaDB not installed.')

Skipped — ChromaDB not installed.


In [6]:
# Production note: intentionally do NOT rely on Chroma's default embedding silently
print('Important production rule:')
print('  → Choose and configure an embedding model deliberately.')
print('  → Use compatible query and document embeddings.')
print('  → Never silently switch models between ingestion and search.')

Important production rule:
  → Choose and configure an embedding model deliberately.
  → Use compatible query and document embeddings.
  → Never silently switch models between ingestion and search.


---
## 📊 6. Understand Retrieval Quality

A vector DB returns **nearest vectors** — but that does NOT guarantee they **answer** the user's question.

### Key metrics

| Metric | What It Measures |
| --- | --- |
| **Precision@K** | Fraction of top-k that are relevant |
| **Recall@K** | Fraction of known relevant items in top-k |
| **MRR** | Rank of the first relevant result |
| **Latency** | Time to complete retrieval |
| **Indexing time** | Time to prepare & store records |
| **Storage cost** | Space for vectors, metadata, indexes |

### Example

Top-5 results contain **3 relevant docs** →  
$$\text{Precision@5} = \frac{3}{5} = 0.60$$

4 relevant docs exist in total, 3 in top-5 →  
$$\text{Recall@5} = \frac{3}{4} = 0.75$$

In [7]:
# Evaluation metrics for retrieval
def precision_at_k(retrieved_ids, relevant_ids, k):
    top = retrieved_ids[:k]
    return sum(1 for i in top if i in relevant_ids) / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    top = retrieved_ids[:k]
    if not relevant_ids:
        return 0.0
    return sum(1 for i in top if i in relevant_ids) / len(relevant_ids)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, i in enumerate(retrieved_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

# Example evaluation
retrieved = ['doc_004', 'doc_001', 'doc_003', 'doc_002']
relevant  = {'doc_001', 'doc_004'}

print(f'Precision@2 = {precision_at_k(retrieved, relevant, 2):.3f}')
print(f'Precision@3 = {precision_at_k(retrieved, relevant, 3):.3f}')
print(f'Recall@3    = {recall_at_k(retrieved, relevant, 3):.3f}')
print(f'MRR         = {reciprocal_rank(retrieved, relevant):.3f}')

Precision@2 = 1.000
Precision@3 = 0.667
Recall@3    = 1.000
MRR         = 1.000


---
## 🆚 7. Choosing a Vector Storage Technology

### ChromaDB
**Best for:** Learning, prototypes, local projects.

- Simple Python workflow
- Convenient local persistence
- Supports metadata & vector retrieval

### Qdrant
**Best for:** Applications needing a dedicated vector service.

- Vector search + payload filtering
- Local & service-based deployments
- Good for advanced retrieval systems

### PostgreSQL + pgvector
**Best for:** Apps already using PostgreSQL that want vectors + relational data together.

- SQL + vector operations
- Relational joins & metadata filtering
- Simplifies architecture when PostgreSQL is already the main DB

### Quick comparison

| Requirement | Suggested start |
| --- | --- |
| Learn vector retrieval quickly | **ChromaDB** |
| Dedicated vector retrieval service | **Qdrant** |
| Combine business data + vectors in PostgreSQL | **pgvector** |
| Exact ID / transactional queries | **Traditional DB** |
| Large-scale retrieval | **Benchmark for your workload** |

> There is no universal winner — test against your **data volume, filters, concurrency, security, latency**.

---
## 🏋️ 8. Practical Project — Knowledge Base Retrieval System

### 🎯 Goal
Build a **searchable knowledge base** that retrieves relevant info from a document collection.

### Functional requirements

- [ ] Ingest **≥ 100 document chunks**
- [ ] Generate & store **compatible embeddings**
- [ ] Associate every chunk with **ID + metadata**
- [ ] Implement **top-k semantic retrieval**
- [ ] Add **metadata filtering**
- [ ] Support **persistent storage**
- [ ] Handle **empty queries / invalid inputs**
- [ ] Add **unit tests** for insert & retrieval
- [ ] Build a **labeled evaluation dataset**
- [ ] Measure **Precision@K, Recall@K, MRR**
- [ ] Document **limitations & security assumptions**

### Suggested advanced features

1. FastAPI search endpoint
2. Batch ingestion
3. Duplicate detection & update handling
4. ChromaDB vs Qdrant comparison
5. Vector search vs keyword search
6. Hybrid retrieval
7. Tenant-aware access control
8. Structured logging + latency metrics

### Definition of done

You can reliably ingest documents, retrieve relevant results, apply filters, persist records, and **show evidence** of search quality.

In [8]:
# Scaffold for the knowledge base retrieval system
import json
from pathlib import Path

class KnowledgeBase:
    """A minimal wrapper around a ChromaDB collection with metadata filtering.

    Real projects should explicitly configure the embedding model and
    keep document/query embeddings compatible."""

    def __init__(self, path, collection_name='kb'):
        import chromadb
        self.client = chromadb.PersistentClient(path=str(path))
        try:
            self.client.delete_collection(collection_name)
        except Exception:
            pass
        self.col = self.client.get_or_create_collection(name=collection_name)

    def ingest(self, ids, texts, metadatas):
        self.col.upsert(ids=ids, documents=texts, metadatas=metadatas)

    def count(self):
        return self.col.count()

    def search(self, query, top_k=3, category=None):
        where = {'category': category} if category else None
        res = self.col.query(query_texts=[query], n_results=top_k, where=where)
        ids = res['ids'][0] or []
        docs = res['documents'][0] or []
        metas = res['metadatas'][0] or []
        dists = res['distances'][0] or []
        return [
            {'id': i, 'document': d, 'metadata': m, 'distance': dist}
            for i, d, m, dist in zip(ids, docs, metas, dists)
        ]

if HAS_CHROMA:
    kb = KnowledgeBase(path=SCRIPT_DIR / 'chroma_kb_demo')
    kb.ingest(
        ids=['t1', 't2', 't3'],
        texts=[
            'Reset your password from account settings.',
            'Billing portal lets you update payment info.',
            'Server troubleshooting guide for engineers.',
        ],
        metadatas=[
            {'category': 'support'},
            {'category': 'billing'},
            {'category': 'engineering'},
        ],
    )
    print('Ingested:', kb.count())
    for r in kb.search('password reset', top_k=2):
        print(r)
else:
    print('Skipped — install chromadb first.')

Skipped — install chromadb first.


---
## ⚠️ 9. Common Mistakes & Best Practices

### ❌ Common mistakes

- Storing vectors **without source text**
- Using **incompatible** embedding models for queries & docs
- **Re-embedding** unchanged docs for every query
- Assuming the **nearest vector is relevant**
- Treating **similarity / distance as probability**
- Ignoring **metadata & access permissions**
- Returning arbitrary numbers of results **without testing**
- Switching models **without rebuilding** vectors
- Choosing a DB by **popularity** instead of requirements

### ✅ Best practices

- Use **stable document IDs**
- Store **source identifiers** + useful metadata
- Keep the **embedding model & config explicit**
- **Batch ingestion** where appropriate
- Measure **retrieval quality & latency**
- Enforce **authorization** in trusted backend code
- Use **realistic evaluation queries**
- Monitor **API failures, storage, retrieval performance**
- **Re-index / re-embed** when necessary
- Keep **vector storage separate** from business authorization decisions

---
## 🎤 10. Interview Questions

**Q1. What is a vector database?**  
A system designed to store vectors and retrieve similar vectors, often with associated text and metadata.

**Q2. Why use a vector DB instead of comparing every vector manually?**  
It provides storage, indexing, filtering, and retrieval that scale better for similarity search.

**Q3. What is approximate nearest-neighbor search?**  
ANN retrieves vectors likely to be close to a query vector, often reducing search time vs. exhaustive search.

**Q4. Vector vs metadata?**  
A vector represents an item numerically. Metadata provides descriptive fields for filtering, organization, and app logic.

**Q5. What is top-k retrieval?**  
It returns up to the **k** highest-ranked candidates according to the retrieval method.

**Q6. Similarity vs distance?**  
Similarity generally increases as items become more alike. Distance generally decreases as vectors get closer.

**Q7. What is metadata filtering?**  
It restricts search candidates using conditions like category, source, department, or date.

**Q8. Can PostgreSQL be used as a vector DB?**  
Yes. PostgreSQL + **pgvector** supports vector storage & similarity search alongside relational data.

**Q9. How do you evaluate retrieval quality?**  
Labeled queries + relevance judgments → Precision@K, Recall@K, MRR. Measure latency separately.

**Q10. Why are vector DBs important for RAG?**  
They retrieve relevant document chunks to provide as context for an LLM.

---
## ✅ 11. Module Completion Checklist

- [ ] Understand vector databases and their purpose
- [ ] Explain **vectors, collections, records, metadata**
- [ ] Compare traditional DBs and vector DBs
- [ ] Understand **similarity metrics & ANN indexes**
- [ ] Insert & update records in ChromaDB
- [ ] Retrieve documents using **semantic queries**
- [ ] Apply **metadata filters**
- [ ] Understand **vector distances & relevance limits**
- [ ] Evaluate **retrieval quality**
- [ ] Complete the **Knowledge Base Retrieval System**
- [ ] Document chosen storage tech & trade-offs

---
## 🏁 12. Key Takeaways

- Embeddings represent text numerically; **vector DBs** make those representations **practical to store and retrieve**
- Retrieval quality depends on **model, dataset, metric, filters, search config**
- **Metadata** improves filtering & organization — but does **not** replace authorization
- **ANN** trades some exactness for performance
- **ChromaDB, Qdrant, pgvector** address different needs
- Retrieval must be **evaluated with relevant data** — not just example similarity scores
- A **reliable retrieval layer** is a major building block for **RAG**

### 🧭 Golden Rule

> **A vector DB returns nearest vectors — not necessarily relevant ones. Store text + metadata together, keep embeddings compatible, enforce authorization in the backend, and evaluate retrieval with labeled queries.**

---

### 🔗 What's Next?

**Module 08 — RAG Fundamentals**

We'll combine **document ingestion, embeddings, retrieval, and LLM generation** into a **Retrieval-Augmented Generation** system.

- Ground model responses in retrieved information
- Provide **source context**
- Reduce reliance on internal model knowledge